[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/80_async_gather_limited_solution.ipynb)

# 🟡 Solution: Async Gather with Concurrency Limit

Reference solution for `async_gather_limited`.

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import asyncio


In [ ]:
# ✅ SOLUTION

async def async_gather_limited(coros, limit: int):
    if limit <= 0:
        raise ValueError("limit must be positive")
    sem = asyncio.Semaphore(limit)

    async def run_one(coro):
        async with sem:
            return await coro

    return await asyncio.gather(*(run_one(coro) for coro in coros))


In [ ]:
# Verify

def run_async(coro):
    import asyncio
    import threading
    box = {}
    def target():
        try:
            box['value'] = asyncio.run(coro)
        except BaseException as e:
            box['error'] = e
    t = threading.Thread(target=target)
    t.start()
    t.join(timeout=5)
    if t.is_alive():
        raise TimeoutError('async test timed out')
    if 'error' in box:
        raise box['error']
    return box.get('value')

async def work(x):
    await asyncio.sleep(0.01)
    return x * x
print(run_async(async_gather_limited([work(i) for i in range(5)], limit=2)))


In [ ]:
# Run judge
from torch_judge import check
check('async_gather_limited')
